In [1]:
import os
import time
import threading
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge
from flask import Flask, render_template, request, send_file

# 1. Folders validation
if not os.path.exists('static'): os.makedirs('static')
if not os.path.exists('templates'): os.makedirs('templates')

# --- DATA COMPILATION ---
CSV_FILE = 'pakistan_economy.csv'
if not os.path.exists(CSV_FILE):
    years = np.arange(2010, 2026)
    dummy_data = {
        'Year': years,
        'GDP': 210 + (years - 2010) * 14 + np.random.normal(0, 4, len(years)),
        'Exports': 22 + (years - 2010) * 1.6 + np.random.normal(0, 0.8, len(years)),
        'Population': 185 + (years - 2010) * 4.2,
        'Inflation': np.random.uniform(9, 24, len(years)),
        'Unemployment': np.random.uniform(4.5, 7.8, len(years))
    }
    pd.DataFrame(dummy_data).to_csv(CSV_FILE, index=False)

df = pd.read_csv(CSV_FILE)
df.columns = df.columns.str.strip()
yr, gdp, exp, pop, inf, unemp = df.columns[0], df.columns[1], df.columns[2], df.columns[3], df.columns[4], df.columns[5]
X = df[[yr, exp, pop]].values

# 3 MULTI-ALGORITHMS INITIALIZED
models_pool = {
    'linear': {'gdp': LinearRegression().fit(X, df[gdp])},
    'rf': {'gdp': RandomForestRegressor(n_estimators=10, random_state=42).fit(X, df[gdp])},
    'ridge': {'gdp': Ridge(alpha=1.0).fit(X, df[gdp])}
}

latest_row = df.iloc[-1]
risk_status = "CRITICAL RISK" if latest_row[inf] > 15 else "STABLE HORIZON"

def build_plots(start=2010, end=2025):
    t_s = int(time.time())
    plt.style.use('dark_background')
    f_df = df[(df[yr] >= start) & (df[yr] <= end)]
    
    # G1: GDP Matrix
    f1, ax1 = plt.subplots(figsize=(11, 3.5))
    ax1.plot(f_df[yr], f_df[gdp], 'o-', color='#00ff88', label='Historical Track')
    ax1.set_title("Gross Domestic Product (GDP) Wave Matrix"); ax1.legend()
    f1.savefig(f'static/g1_{t_s}.png', bbox_inches='tight'); plt.close(f1)
    
    # G2: Inflation Bar Chart
    f2, ax2 = plt.subplots(figsize=(6, 3))
    ax2.bar(f_df[yr], f_df[inf], color='#ff4d4d', alpha=0.8)
    ax2.set_title("CPI Inflation Spikes"); f2.savefig(f'static/g2_{t_s}.png', bbox_inches='tight'); plt.close(f2)
    
    # G3: Heatmap Density
    f3, ax3 = plt.subplots(figsize=(6, 3))
    sns.heatmap(f_df.corr(), annot=True, cmap='mako', fmt=".2f", cbar=False, ax=ax3)
    ax3.set_title("Features Cross Correlation"); f3.savefig(f'static/g3_{t_s}.png', bbox_inches='tight'); plt.close(f3)
    
    # G4: PIE CHART (NEW)
    f4, ax4 = plt.subplots(figsize=(6, 3))
    ax4.pie([22, 54, 18, 6], labels=['Agri', 'Services', 'Mfg', 'Tech'], colors=['#10ac84','#54a0ff','#ff9f43','#5f27cd'], autopct='%1.0f%%')
    ax4.set_title("Sector Share Distribution (%)"); f4.savefig(f'static/g4_{t_s}.png', bbox_inches='tight'); plt.close(f4)

    # G5: STRESS MODEL (NEW)
    f5, ax5 = plt.subplots(figsize=(6, 3))
    ax5.plot(f_df[yr], f_df[gdp], color='#00bfff', marker='o')
    ax5_twin = ax5.twinx()
    ax5_twin.plot(f_df[yr], f_df[inf], color='#ff9f43', marker='x', linestyle='--')
    ax5.set_title("GDP Dynamics vs Inflation Shock Counter-Weight"); f5.savefig(f'static/g5_{t_s}.png', bbox_inches='tight'); plt.close(f5)
    return t_s

current_ts = build_plots()

# HTML WRITER WITH JAVASCRIPT LIVE INTERACTIVE FEATURES
html_home = """<!DOCTYPE html><html><head><link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet"></head>
<body style="background:#08090a; color:#f8f9fa; font-family:'Segoe UI',sans-serif; height:100vh; display:flex; align-items:center; text-center;"><div class="container text-center">
<h1 class="fw-bold mb-3" style="font-size:3.5rem;">Pakistan EconIntelligence Hub <span style="color:#00ff88;">v4.0</span></h1>
<p class="text-secondary mb-4">Enterprise Console with Multi-Model AI Switches and Real-Time Ledger Filters.</p>
<a href="/dashboard" class="btn btn-success btn-lg fw-bold px-5">Launch Interactive Dashboard</a></div></body></html>"""

raw_dashboard_html = """<!DOCTYPE html>
<html>
<head>
    <link href="https://cdn.jsdelivr.net/npm/bootstrap@5.3.0/dist/css/bootstrap.min.css" rel="stylesheet">
    <link rel="stylesheet" href="https://cdn.jsdelivr.net/npm/bootstrap-icons@1.10.0/font/bootstrap-icons.css">
    <style>
        body { background: #0c0e12; color: #e1e7ed; font-family: 'Segoe UI', sans-serif; transition: 0.3s; }
        .sidebar-wrapper { width: 260px; background: #060709; position: fixed; height: 100vh; border-right: 1px solid #1a202c; }
        .main-content-wrapper { margin-left: 260px; padding: 30px; }
        .panel-box { background: #11141d; padding: 20px; border-radius: 12px; border: 1px solid #1e2433; margin-bottom: 24px; }
        .card-stat { background: #11141d; border-radius: 10px; padding: 15px; border: 1px solid #1e2433; }
        img { width: 100%; border-radius: 6px; }
    </style>
    <script>
        // NEW FEATURE 1: LIVE TABLE SEARCH IN JAVASCRIPT
        function dynamicSearchTable() {
            var input = document.getElementById("tableSearchBox");
            var filter = input.value.toUpperCase();
            var table = document.getElementById("ledgerTable");
            var tr = table.getElementsByTagName("tr");
            for (var i = 1; i < tr.length; i++) {
                var td = tr[i].getElementsByTagName("td")[0];
                if (td) {
                    var txtValue = td.textContent || td.innerText;
                    tr[i].style.display = (txtValue.toUpperCase().indexOf(filter) > -1) ? "" : "none";
                }       
            }
        }
        // NEW FEATURE 2: LIVE SIDEBAR THEME SWITCHER
        function triggerThemeShift() {
            var el = document.getElementById("bodyTagSelector");
            if(el.style.backgroundColor === "rgb(12, 14, 18)" || el.style.backgroundColor === "") {
                el.style.backgroundColor = "#021c12"; // Cyber Emerald
            } else { el.style.backgroundColor = "#0c0e12"; }
        }
    </script>
</head>
<body id="bodyTagSelector">

    <div class="sidebar-wrapper d-flex flex-column justify-content-between py-4">
        <div class="px-3">
            <h5 class="fw-bold text-white"><i class="bi bi-cpu text-success me-2"></i>EconIntel Console</h5>
            <hr class="border-secondary">
            <button onclick="triggerThemeShift()" class="btn btn-sm btn-info text-dark fw-bold w-100 mb-3"><i class="bi bi-palette-fill me-2"></i>Switch Cyber Green</button>
            
            <div class="p-3 bg-dark bg-opacity-50 rounded border border-danger text-muted small">
                <span class="text-danger fw-bold d-block mb-1"><i class="bi bi-bug-fill me-1"></i>ANOMALY LOG ENGINE:</span>
                Outlier identified during high inflationary index loops. Risk index calibrated.
            </div>
        </div>
        <div class="px-3">
            <a href="/download_txt_report" class="btn btn-sm btn-warning w-100 fw-bold mb-2"><i class="bi bi-file-earmark-text me-1"></i>Download TXT Report</a>
            <a href="/download" class="btn btn-sm btn-success w-100 fw-bold"><i class="bi bi-download me-1"></i>Download CSV Dataset</a>
        </div>
    </div>

    <div class="main-content-wrapper">
        <div class="row g-3 mb-4">
            <div class="col-md-3"><div class="card-stat" style="border-left:4px solid #00ff88;"><small class="text-muted d-block">GDP BASELINE VOLUME</small><h4 class="fw-bold mt-1">VAR_LATEST_GDP B</h4></div></div>
            <div class="col-md-3"><div class="card-stat" style="border-left:4px solid #ff9f43;"><small class="text-muted d-block">CPI CORE INFLATION</small><h4 class="fw-bold mt-1">VAR_LATEST_INF%</h4></div></div>
            <div class="col-md-3"><div class="card-stat" style="border-left:4px solid #00d2d3;"><small class="text-muted d-block">UNEMPLOYMENT EQUILIBRIUM</small><h4 class="fw-bold mt-1">VAR_LATEST_UNEMP%</h4></div></div>
            <div class="col-md-3"><div class="card-stat" style="border-left:4px solid #ff4d4d;"><small class="text-muted d-block">RISK LEVEL INDEX</small><h5 class="fw-bold text-danger mt-1">VAR_RISK_INDEX</h5></div></div>
        </div>

        <div class="panel-box border border-success border-opacity-25">
            <form action="/filter" method="post" class="row g-3 align-items-center">
                <div class="col-md-4">
                    <label class="small text-muted fw-bold">START TIMELINE BOUNDARY</label>
                    <select name="start_yr" class="form-select bg-dark text-white border-secondary"><option value="2010">2010 Baseline</option><option value="2015">2015 Midterm</option></select>
                </div>
                <div class="col-md-4">
                    <label class="small text-muted fw-bold">END TIMELINE BOUNDARY</label>
                    <select name="end_yr" class="form-select bg-dark text-white border-secondary"><option value="2025" selected>2025 Current Log</option><option value="2020">2020 Previous Log</option></select>
                </div>
                <div class="col-md-4 align-self-end"><button class="btn btn-success fw-bold w-100">Re-Calibrate Visual Graphs</button></div>
            </form>
        </div>

        <div class="row"><div class="col-md-12"><div class="panel-box"><h6>1. Macroeconomic Gross Domestic Product (GDP) Development Matrix</h6><img src="/static/g1_VAR_TIMESTAMP.png"></div></div></div>
        <div class="row">
            <div class="col-md-6"><div class="panel-box"><h6>2. Consumer Price Index Velocity (CPI Bars)</h6><img src="/static/g2_VAR_TIMESTAMP.png"></div></div>
            <div class="col-md-6"><div class="panel-box"><h6>3. Multi-Variable Pearson Structural Heatmap</h6><img src="/static/g3_VAR_TIMESTAMP.png"></div></div>
        </div>
        <div class="row mb-4">
            <div class="col-md-6"><div class="panel-box"><h6><span class="badge bg-primary me-2">NEW CHART</span>4. Macro Sector Share Component Allocation</h6><img src="/static/g4_VAR_TIMESTAMP.png"></div></div>
            <div class="col-md-6"><div class="panel-box"><h6><span class="badge bg-primary me-2">NEW CHART</span>5. National Stress Index Model: GDP vs Inflation Resistance</h6><img src="/static/g5_VAR_TIMESTAMP.png"></div></div>
        </div>

        <div class="row">
            <div id="predictor" class="col-md-4">
                <div class="panel-box border border-warning border-opacity-25">
                    <h5 class="text-warning mb-3 fw-bold"><i class="bi bi-cpu-fill me-2"></i>Prediction Simulator Lab</h5>
                    <form action="/predict#predictor" method="post">
                        <label class="small text-muted fw-bold">SELECT MACHINE LEARNING ENGINE:</label>
                        <select name="ai_model" class="form-select bg-dark text-white border-secondary mb-2">
                            <option value="linear" {% if selected_model == 'linear' %}selected{% endif %}>Linear Regression Topology</option>
                            <option value="rf" {% if selected_model == 'rf' %}selected{% endif %}>Random Forest Ensembles</option>
                            <option value="ridge" {% if selected_model == 'ridge' %}selected{% endif %}>Ridge Regularized Matrix</option>
                        </select>
                        <label class="small text-muted fw-bold">TARGET EVALUATION YEAR:</label>
                        <input type="number" name="y" value="{{ y_v if y_v else '2027' }}" class="form-control bg-dark text-white border-secondary mb-2" required>
                        <label class="small text-muted fw-bold">SIMULATED EXPORTS VOLUME ($ B):</label>
                        <input type="number" step="0.1" name="e" value="{{ e_v if e_v else '34.0' }}" class="form-control bg-dark text-white border-secondary mb-2" required>
                        <label class="small text-muted fw-bold">TARGET DEMOGRAPHICS CAPACITY (M):</label>
                        <input type="number" step="0.1" name="p" value="{{ p_v if p_v else '255.0' }}" class="form-control bg-dark text-white border-secondary mb-3" required>
                        <button class="btn btn-warning w-100 fw-bold">Compute Algorithmic Forecast</button>
                    </form>
                    {% if res_g %}
                    <div class="mt-3 pt-3 border-top border-secondary text-center">
                        <small class="text-muted d-block">COMPUTED FORECAST GDP VALUE:</small><h3 class="text-success fw-bold m-0">{{ res_g }}</h3>
                        <small class="text-muted d-block mt-2">ACTIVE AI CORE ENGINE:</small><span class="badge bg-secondary text-uppercase">{{ selected_model }} Rig</span>
                    </div>
                    {% endif %}
                </div>
            </div>

            <div class="col-md-8">
                <div class="panel-box">
                    <div class="d-flex justify-content-between align-items-center mb-2">
                        <h5 class="text-success fw-bold m-0"><i class="bi bi-database-fill me-2"></i>Micro-Log Ledger</h5>
                        <input type="text" id="tableSearchBox" onkeyup="dynamicSearchTable()" placeholder="🔍 Fast Live Search by Year..." class="form-control form-control-sm bg-dark text-white border-secondary w-50">
                    </div>
                    <div class="table-responsive" style="max-height:340px; overflow-y:auto;" id="ledgerTable">
                        {{ table_html|safe }}
                    </div>
                </div>
            </div>
        </div>
    </div>
</body>
</html>"""

formatted_dashboard = raw_dashboard_html.replace("VAR_LATEST_YEAR", str(int(latest_row[yr]))).replace("VAR_LATEST_GDP", str(f"{latest_row[gdp]:.2f}")).replace("VAR_LATEST_INF", str(f"{latest_row[inf]:.1f}")).replace("VAR_LATEST_UNEMP", str(f"{latest_row[unemp]:.1f}")).replace("VAR_TIMESTAMP", str(current_ts)).replace("VAR_RISK_INDEX", risk_status)
with open('templates/home.html', 'w', encoding='utf-8') as f: f.write(html_home)
with open('templates/dashboard.html', 'w', encoding='utf-8') as f: f.write(formatted_dashboard)

# --- Flask Server Architecture ---
app = Flask(__name__)

@app.route('/')
def home(): return render_template('home.html')

@app.route('/dashboard')
def dashboard():
    table_html = df.to_html(classes='table table-dark table-striped table-hover m-0 text-center small align-middle', index=False)
    return render_template('dashboard.html', y_v='', e_v='', p_v='', table_html=table_html, res_g=None, selected_model='linear')

@app.route('/filter', methods=['POST'])
def filter_timeline():
    try:
        s, e = int(request.form.get('start_yr', 2010)), int(request.form.get('end_yr', 2025))
        new_ts = build_plots(s, e)
        updated_db = raw_dashboard_html.replace("VAR_LATEST_YEAR", str(int(latest_row[yr]))).replace("VAR_LATEST_GDP", str(f"{latest_row[gdp]:.2f}")).replace("VAR_LATEST_INF", str(f"{latest_row[inf]:.1f}")).replace("VAR_LATEST_UNEMP", str(f"{latest_row[unemp]:.1f}")).replace("VAR_TIMESTAMP", str(new_ts)).replace("VAR_RISK_INDEX", risk_status)
        with open('templates/dashboard.html', 'w', encoding='utf-8') as f: f.write(updated_db)
        table_html = df.to_html(classes='table table-dark table-striped table-hover m-0 text-center small align-middle', index=False)
        return render_template('dashboard.html', y_v='', e_v='', p_v='', table_html=table_html, res_g=None, selected_model='linear')
    except Exception as ex: return str(ex)

@app.route('/predict', methods=['POST'])
def predict():
    try:
        m_type = request.form.get('ai_model', 'linear')
        y_v, e_v, p_v = int(request.form['y']), float(request.form['e']), float(request.form['p'])
        selected_model_rig = models_pool[m_type]['gdp']
        rg = selected_model_rig.predict([[y_v, e_v, p_v]])[0]
        table_html = df.to_html(classes='table table-dark table-striped table-hover m-0 text-center small align-middle', index=False)
        return render_template('dashboard.html', y_v=y_v, e_v=e_v, p_v=p_v, table_html=table_html, res_g=f"${rg:.2f} Billion", selected_model=m_type)
    except Exception as err: return str(err)

@app.route('/download_txt_report')
def download_txt_report():
    report_path = 'executive_intelligence_report.txt'
    with open(report_path, 'w') as f:
        f.write("==============================================\n   PAKISTAN ECONOMY ANALYTICAL REPORT   \n==============================================\n")
        f.write(f"Latest Target GDP Level: ${latest_row[gdp]:.2f} Billion\nCore Inflation Index: {latest_row[inf]:.1f}%\nRisk Profile Status: {risk_status}\n")
    return send_file(report_path, as_attachment=True)

@app.route('/download')
def download(): return send_file(CSV_FILE, as_attachment=True)

# THREAD DEPLOYMENT
def init_web_instance():
    app.run(port=5200, debug=False, use_reloader=False)  # CHANGED PORT TO 5200 FOR FRESH RELOAD

threading.Thread(target=init_web_instance, daemon=True).start()
print("\n✅ SEED LEVEL STACK SUCCESSFUL: Server background mein chalu hai.")
print("👉 Is bilkul naye address ko open karein: http://127.0.0.1:5200")


✅ SEED LEVEL STACK SUCCESSFUL: Server background mein chalu hai.
👉 Is bilkul naye address ko open karein: http://127.0.0.1:5200
 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5200
Press CTRL+C to quit
127.0.0.1 - - [18/Jun/2026 04:21:25] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:25] "GET /favicon.ico HTTP/1.1" 404 -
127.0.0.1 - - [18/Jun/2026 04:21:30] "GET /dashboard HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:31] "GET /static/g2_1781738471.png HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:31] "GET /static/g1_1781738471.png HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:31] "GET /static/g3_1781738471.png HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:31] "GET /static/g4_1781738471.png HTTP/1.1" 200 -
127.0.0.1 - - [18/Jun/2026 04:21:31] "GET /static/g5_1781738471.png HTTP/1.1" 200 -
